CHECK DATABASE INTEGRITY AND TABLES

In [18]:
import sqlite3
import pandas as pd
import os
from pathlib import Path
# current_dir = Path(__file__).resolve().parent #not working in jupyter notebook
current_dir = Path(os.getcwd())

parent_dir = current_dir.parent
db_path = parent_dir / "data" / "Data Engineer_ETL Assignment.db"

conn = sqlite3.connect(db_path)
try:
    conn.execute("SELECT 1")
    print("Connection to the database was successful.")
except sqlite3.Error as e:
    print(f"An error occurred: {e}")

conn.close()
# print(tables)

An error occurred: database disk image is malformed


TRY : Database Recovery, downloading sqlite3.exe to current directory(tests) and run recovery

In [21]:
import sqlite3
import os
import logging

SOURCE_DB = db_path
RECOVERED_DB = parent_dir / "data" / "recovered_database.db"
DUMP_FILE = parent_dir / "data" / "db_dump.sql"


logging.basicConfig(level=logging.INFO,format="%(asctime)s - %(levelname)s - %(message)s")


def check_integrity(db_path):
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        cursor.execute("PRAGMA integrity_check;")
        result = cursor.fetchall()
        conn.close()

        if result[0][0] == "ok":
            logging.info("Integrity OK.")
            return True
        else:
            logging.warning("Database corruption detected.")
            return False

    except Exception as e:
        logging.error(f"Integrity check failed: {e}")
        return False


def dump_database(db_path, dump_path):
    try:
        conn = sqlite3.connect(db_path)

        with open(dump_path, "w", encoding="utf-8") as f:
            for line in conn.iterdump():
                f.write(f"{line}\n")

        conn.close()

        logging.info("Database dump created successfully.")
        return True

    except Exception as e:
        logging.error(f"Database dump failed: {e}")
        return False


def rebuild_database(dump_path, new_db_path):
    try:
        if os.path.exists(new_db_path):
            os.remove(new_db_path)

        conn = sqlite3.connect(new_db_path)

        with open(dump_path, "r", encoding="utf-8") as f:
            sql_script = f.read()

        conn.executescript(sql_script)
        conn.commit()
        conn.close()

        logging.info("Recovered database created successfully.")

    except Exception as e:
        logging.error(f"Database rebuild failed: {e}")


def main():
    logging.info("Starting SQLite recovery process")

    integrity_ok = check_integrity(SOURCE_DB)

    if integrity_ok:
        logging.info("Database not corrupted. Recovery not required.")
        return

    dump_success = dump_database(SOURCE_DB, DUMP_FILE)

    if dump_success:
        rebuild_database(DUMP_FILE, RECOVERED_DB)
    else:
        logging.error("Recovery failed. Dump could not be created.")

main()

2026-03-14 19:21:06,386 - INFO - Starting SQLite recovery process
2026-03-14 19:21:06,388 - ERROR - Integrity check failed: database disk image is malformed
2026-03-14 19:21:06,390 - ERROR - Database dump failed: database disk image is malformed
2026-03-14 19:21:06,391 - ERROR - Recovery failed. Dump could not be created.
